# Notebook 06: Multi-Agent Clinical Synthesis
---
This notebook orchestrates the final stage of the clinical pipeline: generating a structured, evidence-based report using three specialized AI agents.

### ⚠️ Troubleshooting NameError
If you see `NameError: name 'ExplanationAgent' is not defined`, please ensure you have run the **Initialization** cell below first.

In [1]:
import sys
import os
from pathlib import Path

# 1. Dependency Check
try:
    import huggingface_hub
    import dotenv
    print('✅ Required libraries (huggingface_hub, python-dotenv) found.')
except ImportError:
    print('❌ Missing libraries! Run: !pip install huggingface-hub python-dotenv')

# 2. Setup Path to import from src
current_dir = Path(os.getcwd())
project_root = current_dir.parent if current_dir.name == 'notebooks' else current_dir
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

import config
from src.agents.explanation_agent import ExplanationAgent
from src.agents.validation_agent import ValidationAgent
from src.agents.summary_agent import SummaryAgent

print('✅ Code imports successful.')

# 3. Initialize Agents
explanation_agent = ExplanationAgent()
validation_agent  = ValidationAgent()
summary_agent     = SummaryAgent()

print('✅ All Agents initialized correctly.')

✅ Required libraries (huggingface_hub, python-dotenv) found.
✅ Code imports successful.
✅ All Agents initialized correctly.


In [2]:
# Demo Case: MPX1072_synpic34899
case_id         = 'MPX1072_synpic34899'
predicted_label = 'Infection_Inflammatory'
confidence      = 0.9888
shap_tokens     = ['thyroiditis', 'hashimoto', 'dryness', 'irritation', 'lid retraction', 'blurry', 'progressing']
gradcam_region  = 'upper-left quadrant (sinus/nasal cavity)'
rag_retrievals  = [
    {'rank':1, 'sim':0.7251, 'label':'Other',                  'id':'MPX2233_synpic17324'},
    {'rank':2, 'sim':0.7250, 'label':'Infection_Inflammatory', 'id':'MPX1957_synpic41671'},
    {'rank':3, 'sim':0.7142, 'label':'Clinical Sign',          'id':'MPX1628_synpic41589'},
    {'rank':4, 'sim':0.7115, 'label':'Other',                  'id':'MPX2391_synpic17893'},
    {'rank':5, 'sim':0.7114, 'label':'Other',                  'id':'MPX2433_synpic48296'},
]

patient_synopsis = "A middle-aged patient presenting with progressive eye irritation, dryness, and upper eyelid retraction. Recent clinical note mentions 'hashimoto' and 'thyroiditis'. Imaging shows inflammatory changes."

print(f'Patient Case: {case_id}')
print(f'Prediction: {predicted_label} ({confidence:.2%})')

Patient Case: MPX1072_synpic34899
Prediction: Infection_Inflammatory (98.88%)


In [3]:
print('--- AGENT 1: EXPLANATION ---')
explanation_output = explanation_agent.generate_reasoning(
    diagnosis=predicted_label,
    confidence=confidence,
    shap_tokens=shap_tokens,
    gradcam_region=gradcam_region
)
print(explanation_output)

--- AGENT 1: EXPLANATION ---
The model predicted an Infection_Inflammatory diagnosis with high confidence, primarily influenced by the clinical terms "thyroiditis" and "hashimoto," which suggest an underlying autoimmune condition that can present with inflammatory symptoms. The presence of "dryness," "irritation," "lid retraction," and "blurry" further supports this, indicating possible ocular manifestations often seen in autoimmune thyroid disorders. Additionally, the Grad-CAM hotspot in the upper-left quadrant of the image, corresponding to the sinus/nasal cavity, suggests localized inflammation or infection, which may be contributing to the overall inflammatory state.


In [4]:
print('--- AGENT 2: VALIDATION ---')
validation_output = validation_agent.validate_prediction(
    predicted_label=predicted_label,
    retrieved_cases=rag_retrievals
)
print(validation_output)

--- AGENT 2: VALIDATION ---
### Analysis of Prediction Consistency with Historical Cases

#### 1. Consistency with High-Similarity Cases
- **Prediction**: Infection_Inflammatory
- **Historical Cases**:
  - Case MPX2233_synpic17324 (0.00% sim): Diagnosed as 'Other'
  - Case MPX1957_synpic41671 (0.00% sim): Diagnosed as 'Infection_Inflammatory'
  - Case MPX1628_synpic41589 (0.00% sim): Diagnosed as 'Clinical Sign'
  - Case MPX2391_synpic17893 (0.00% sim): Diagnosed as 'Other'
  - Case MPX2433_synpic48296 (0.00% sim): Diagnosed as 'Other'

**Observation**:
- All five historical cases have a similarity score of 0.00%, indicating no significant similarity to the current case.
- Among these cases, only one case (MPX1957_synpic41671) has the same diagnosis as the predicted diagnosis ('Infection_Inflammatory').

#### 2. Label Shift Cases
- **Label Shift**: A label shift occurs when a case with high similarity has a different diagnosis.
- **Analysis**:
  - Since all cases have a similarity scor

In [5]:
print('--- AGENT 3: SUMMARY ---')
final_report = summary_agent.generate_report(
    patient_synopsis=patient_synopsis,
    diagnosis=predicted_label,
    confidence=confidence,
    clinical_reasoning=explanation_output,
    validation_report=validation_output
)
print(final_report)

--- AGENT 3: SUMMARY ---
## 🏥 Patient Summary
The patient is a middle-aged individual presenting with progressive eye irritation, dryness, and upper eyelid retraction. Recent clinical notes mention "Hashimoto's thyroiditis." Imaging reveals inflammatory changes.

## 🔬 Predicted Diagnosis & Confidence
The AI model predicts an **Infection_Inflammatory** diagnosis with a high confidence level of **98.88%**.

## 🧠 Clinical Reasoning (XAI)
The model's high confidence in the Infection_Inflammatory diagnosis is primarily influenced by the clinical terms "thyroiditis" and "Hashimoto," which suggest an underlying autoimmune condition known to cause inflammatory symptoms. The presence of "dryness," "irritation," "lid retraction," and "blurry vision" further supports this diagnosis, as these symptoms are often associated with ocular manifestations of autoimmune thyroid disorders. Additionally, the Grad-CAM heatmap highlights the upper-left quadrant of the image, corresponding to the sinus/nasal c

In [6]:
from IPython.display import display, Markdown
display(Markdown(f'# 🏥 Clinical Review Report: {case_id}'))
display(Markdown(final_report))

# 🏥 Clinical Review Report: MPX1072_synpic34899

## 🏥 Patient Summary
The patient is a middle-aged individual presenting with progressive eye irritation, dryness, and upper eyelid retraction. Recent clinical notes mention "Hashimoto's thyroiditis." Imaging reveals inflammatory changes.

## 🔬 Predicted Diagnosis & Confidence
The AI model predicts an **Infection_Inflammatory** diagnosis with a high confidence level of **98.88%**.

## 🧠 Clinical Reasoning (XAI)
The model's high confidence in the Infection_Inflammatory diagnosis is primarily influenced by the clinical terms "thyroiditis" and "Hashimoto," which suggest an underlying autoimmune condition known to cause inflammatory symptoms. The presence of "dryness," "irritation," "lid retraction," and "blurry vision" further supports this diagnosis, as these symptoms are often associated with ocular manifestations of autoimmune thyroid disorders. Additionally, the Grad-CAM heatmap highlights the upper-left quadrant of the image, corresponding to the sinus/nasal cavity, indicating localized inflammation or infection that may contribute to the overall inflammatory state.

## 📁 Historical Case Validation
### Analysis of Prediction Consistency with Historical Cases
- **Prediction**: Infection_Inflammatory
- **Historical Cases**:
  - Case MPX2233_synpic17324 (0.00% sim): Diagnosed as 'Other'
  - Case MPX1957_synpic41671 (0.00% sim): Diagnosed as 'Infection_Inflammatory'
  - Case MPX1628_synpic41589 (0.00% sim): Diagnosed as 'Clinical Sign'
  - Case MPX2391_synpic17893 (0.00% sim): Diagnosed as 'Other'
  - Case MPX2433_synpic48296 (0.00% sim): Diagnosed as 'Other'

**Observation**:
- All five historical cases have a similarity score of 0.00%, indicating no significant similarity to the current case.
- Only one case (MPX1957_synpic41671) matches the predicted diagnosis of 'Infection_Inflammatory'.

### Final Validation Verdict
- **Consistency**: The prediction is not consistent with the majority of the historical cases, as only one out of five cases matches the predicted diagnosis.
- **Support Level**: Not strongly supported by historical data due to the lack of similar cases.

## ⚠️ Caveats & Limitations
- **Lack of Similar Historical Cases**: The absence of high-similarity cases limits the validation of the model's prediction.
- **Autoimmune Complexity**: The diagnosis of autoimmune conditions like Hashimoto's thyroiditis can be complex and may require additional clinical and laboratory evaluations.
- **Imaging Interpretation**: The Grad-CAM heatmap provides a visual aid but should be interpreted in conjunction with clinical findings and other diagnostic tests.

## 📋 Recommended Next Steps
1. **Comprehensive Ophthalmologic Evaluation**: Conduct a detailed ophthalmologic examination to assess the extent of ocular involvement.
2. **Thyroid Function Tests**: Order thyroid function tests (TSH, T3, T4, and anti-thyroid peroxidase antibodies) to confirm or rule out Hashimoto's thyroiditis.
3. **Referral to a Specialist**: Consider referral to an endocrinologist for further management of thyroid-related issues.
4. **Follow-Up Imaging**: If necessary, repeat imaging studies to monitor the progression of inflammatory changes.
5. **Patient Education**: Educate the patient about the potential causes and management of their symptoms, including the importance of regular follow-up.